In [11]:
import math
import logging
import time
import sys
import argparse
import torch
import numpy as np
import pickle
from pathlib import Path

from tgn.evaluation.evaluation import eval_edge_prediction
from tgn.model.tgn import TGN
from tgn.utils.utils import EarlyStopMonitor, RandEdgeSampler, get_neighbor_finder
from tgn.utils.data_processing import get_data, compute_time_statistics

import importlib
import sys
importlib.reload(sys.modules['tgn.utils.data_processing'])
import importlib
import tgn.model.tgn as tgn
importlib.reload(tgn)
TGN = tgn.TGN
torch.manual_seed(0)
np.random.seed(0)



BATCH_SIZE = 256
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_EPOCH = 100
NUM_HEADS = 4
DROP_OUT = 0.1
data = "p0.8_mu0.1_1"
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 128
TIME_DIM = 100
USE_MEMORY = False
MESSAGE_DIM = 128
MEMORY_DIM = 128
uniform = False 
prefix = ""
backprop_every = 1
message_function = 'mlp'
use_source_embedding_in_message= True
use_destination_embedding_in_message = True 
aggregator = 'mean'
different_new_nodes = False
randomize_features = True
n_runs = 1 
memory_update_at_end = False 
embedding_module = 'graph_attention' # graph_attention, graph_sum, time, identity
dyrep = True 
memory_updater = "gru"
patience = 10
bidirection = True

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{data}-{epoch}.pth'

### set up logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)

### Extract data for training, validation and testing
node_features, edge_features, full_data, train_data, val_data, test_data, new_node_val_data, \
new_node_test_data = get_data(data,
                              different_new_nodes_between_val_and_test=different_new_nodes, randomize_features=randomize_features)

# Initialize training neighbor finder to retrieve temporal graph
train_ngh_finder = get_neighbor_finder(train_data, uniform)

# Initialize validation and test neighbor finder to retrieve temporal graph
full_ngh_finder = get_neighbor_finder(full_data, uniform)

# Initialize negative samplers. Set seeds for validation and testing so negatives are the same
# across different runs
# NB: in the inductive setting, negatives are sampled only amongst other new nodes
train_rand_sampler = RandEdgeSampler(train_data.sources, train_data.destinations)
val_rand_sampler = RandEdgeSampler(full_data.sources, full_data.destinations, seed=0)
nn_val_rand_sampler = RandEdgeSampler(new_node_val_data.sources, new_node_val_data.destinations, seed=1)
test_rand_sampler = RandEdgeSampler(full_data.sources, full_data.destinations, seed=2)
nn_test_rand_sampler = RandEdgeSampler(new_node_test_data.sources,
                                       new_node_test_data.destinations,
                                       seed=3)

device = 'cuda' if torch.cuda.is_available() else 'mps'

# Compute time statistics
mean_time_shift_src, std_time_shift_src, mean_time_shift_dst, std_time_shift_dst = \
  compute_time_statistics(full_data.sources, full_data.destinations, full_data.timestamps)

for i in range(n_runs):
  results_path = "results/{}_{}.pkl".format(prefix, i) if i > 0 else "results/{}.pkl".format(prefix)
  Path("results/").mkdir(parents=True, exist_ok=True)

  # Initialize Model
  tgn = TGN(neighbor_finder=train_ngh_finder, node_features=node_features,
            edge_features=edge_features, device=device,
            n_layers=NUM_LAYER,
            n_heads=NUM_HEADS, dropout=DROP_OUT, use_memory=USE_MEMORY,
            message_dimension=MESSAGE_DIM, memory_dimension=MEMORY_DIM,
            memory_update_at_start=not memory_update_at_end,
            embedding_module_type=embedding_module,
            message_function=message_function,
            aggregator_type=aggregator,
            memory_updater_type=memory_updater,
            n_neighbors=NUM_NEIGHBORS,
            mean_time_shift_src=mean_time_shift_src, std_time_shift_src=std_time_shift_src,
            mean_time_shift_dst=mean_time_shift_dst, std_time_shift_dst=std_time_shift_dst,
            use_destination_embedding_in_message=use_destination_embedding_in_message,
            use_source_embedding_in_message=use_source_embedding_in_message,
            dyrep=dyrep, bidirection=bidirection)


  criterion = torch.nn.BCELoss()
  optimizer = torch.optim.AdamW(tgn.parameters(), lr=LEARNING_RATE)
  tgn = tgn.to(device)

  num_instance = len(train_data.sources)
  num_batch = math.ceil(num_instance / BATCH_SIZE)

  logger.info('num of training instances: {}'.format(num_instance))
  logger.info('num of batches per epoch: {}'.format(num_batch))
  idx_list = np.arange(num_instance)

  new_nodes_val_aps = []
  val_aps = []
  epoch_times = []
  total_epoch_times = []
  train_losses = []

  early_stopper = EarlyStopMonitor(max_round=patience)
  for epoch in range(NUM_EPOCH):
    start_epoch = time.time()
    ### Training

    # Reinitialize memory of the model at the start of each epoch
    if USE_MEMORY:
      tgn.memory.__init_memory__()

    # Train using only training graph
    tgn.set_neighbor_finder(train_ngh_finder)
    m_loss = []

    logger.info('start {} epoch'.format(epoch))
    for k in range(0, num_batch, backprop_every):
      loss = 0
      optimizer.zero_grad()

      # Custom loop to allow to perform backpropagation only every a certain number of batches
      for j in range(backprop_every):
        batch_idx = k + j

        if batch_idx >= num_batch:
          continue

        start_idx = batch_idx * BATCH_SIZE
        end_idx = min(num_instance, start_idx + BATCH_SIZE)
        sources_batch, destinations_batch = train_data.sources[start_idx:end_idx], \
                                            train_data.destinations[start_idx:end_idx]
        edge_idxs_batch = train_data.edge_idxs[start_idx: end_idx]
        timestamps_batch = train_data.timestamps[start_idx:end_idx]

        size = len(sources_batch)
        _, negatives_batch = train_rand_sampler.sample(size)

        with torch.no_grad():
          pos_label = torch.ones(size, dtype=torch.float, device=device)
          neg_label = torch.zeros(size, dtype=torch.float, device=device)

        tgn = tgn.train()
        pos_prob, neg_prob = tgn.compute_edge_probabilities(sources_batch, destinations_batch, negatives_batch, timestamps_batch, edge_idxs_batch, NUM_NEIGHBORS)

        loss += criterion(pos_prob.squeeze(), pos_label) + criterion(neg_prob.squeeze(), neg_label)

      loss /= backprop_every

      loss.backward()
      optimizer.step()
      m_loss.append(loss.item())

      # Detach memory after 'backprop_every' number of batches so we don't backpropagate to
      # the start of time
      if USE_MEMORY:
        tgn.memory.detach_memory()

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)

    ### Validation
    # Validation uses the full graph
    tgn.set_neighbor_finder(full_ngh_finder)

    if USE_MEMORY:
      # Backup memory at the end of training, so later we can restore it and use it for the
      # validation on unseen nodes
      train_memory_backup = tgn.memory.backup_memory()

    val_ap, val_auc = eval_edge_prediction(model=tgn,
                                            negative_edge_sampler=val_rand_sampler,
                                            data=val_data,
                                            n_neighbors=NUM_NEIGHBORS)
    if USE_MEMORY:
      val_memory_backup = tgn.memory.backup_memory()
      # Restore memory we had at the end of training to be used when validating on new nodes.
      # Also backup memory after validation so it can be used for testing (since test edges are
      # strictly later in time than validation edges)
      tgn.memory.restore_memory(train_memory_backup)

    # Validate on unseen nodes
    nn_val_ap, nn_val_auc = eval_edge_prediction(model=tgn,
                                                negative_edge_sampler=val_rand_sampler,
                                                data=new_node_val_data,
                                                n_neighbors=NUM_NEIGHBORS)

    if USE_MEMORY:
      # Restore memory we had at the end of validation
      tgn.memory.restore_memory(val_memory_backup)

    new_nodes_val_aps.append(nn_val_ap)
    val_aps.append(val_ap)
    train_losses.append(np.mean(m_loss))

    # Save temporary results to disk
    pickle.dump({
      "val_aps": val_aps,
      "new_nodes_val_aps": new_nodes_val_aps,
      "train_losses": train_losses,
      "epoch_times": epoch_times,
      "total_epoch_times": total_epoch_times
    }, open(results_path, "wb"))

    total_epoch_time = time.time() - start_epoch
    total_epoch_times.append(total_epoch_time)

    logger.info('epoch: {} took {:.2f}s'.format(epoch, total_epoch_time))
    logger.info('Epoch mean loss: {}'.format(np.mean(m_loss)))
    logger.info(
      'val auc: {}, new node val auc: {}'.format(val_auc, nn_val_auc))
    logger.info(
      'val ap: {}, new node val ap: {}'.format(val_ap, nn_val_ap))

    # Early stopping
    if early_stopper.early_stop_check(val_ap):
      logger.info('No improvement over {} epochs, stop training'.format(early_stopper.max_round))
      logger.info(f'Loading the best model at epoch {early_stopper.best_epoch}')
      best_model_path = get_checkpoint_path(early_stopper.best_epoch)
      tgn.load_state_dict(torch.load(best_model_path))
      logger.info(f'Loaded the best model at epoch {early_stopper.best_epoch} for inference')
      tgn.eval()
      break
    else:
      torch.save(tgn.state_dict(), get_checkpoint_path(epoch))

  # Training has finished, we have loaded the best model, and we want to backup its current
  # memory (which has seen validation edges) so that it can also be used when testing on unseen
  # nodes
  if USE_MEMORY:
    val_memory_backup = tgn.memory.backup_memory()

  ### Test
  tgn.embedding_module.neighbor_finder = full_ngh_finder
  test_ap, test_auc = eval_edge_prediction(model=tgn,
                                            negative_edge_sampler=test_rand_sampler,
                                            data=test_data,
                                            n_neighbors=NUM_NEIGHBORS)

  if USE_MEMORY:
    tgn.memory.restore_memory(val_memory_backup)

  # Test on unseen nodes
  nn_test_ap, nn_test_auc = eval_edge_prediction(model=tgn,
                                                  negative_edge_sampler=nn_test_rand_sampler,
                                                  data=new_node_test_data,
                                                  n_neighbors=NUM_NEIGHBORS)

  logger.info(
    'Test statistics: Old nodes -- auc: {}, ap: {}'.format(test_auc, test_ap))
  logger.info(
    'Test statistics: New nodes -- auc: {}, ap: {}'.format(nn_test_auc, nn_test_ap))
  # Save results for this run
  pickle.dump({
    "val_aps": val_aps,
    "new_nodes_val_aps": new_nodes_val_aps,
    "test_ap": test_ap,
    "new_node_test_ap": nn_test_ap,
    "epoch_times": epoch_times,
    "train_losses": train_losses,
    "total_epoch_times": total_epoch_times
  }, open(results_path, "wb"))

  logger.info('Saving TGN model')
  if USE_MEMORY:
    # Restore memory at the end of validation (save a model which is ready for testing)
    tgn.memory.restore_memory(val_memory_backup)
  torch.save(tgn.state_dict(), MODEL_SAVE_PATH)
  logger.info('TGN model saved')


The dataset has 472 interactions, involving 50 different nodes
The training dataset has 274 interactions, involving 45 different nodes
The validation dataset has 71 interactions, involving 48 different nodes
The test dataset has 71 interactions, involving 48 different nodes
The new node validation dataset has 14 interactions, involving 14 different nodes
The new node test dataset has 7 interactions, involving 11 different nodes
5 nodes were used for the inductive testing, i.e. are never seen during training


INFO:root:num of training instances: 274
INFO:root:num of batches per epoch: 2
INFO:root:start 0 epoch
INFO:root:epoch: 0 took 5.58s
INFO:root:Epoch mean loss: 1.4049487113952637
INFO:root:val auc: 0.534913707597699, new node val auc: 0.6887755102040816
INFO:root:val ap: 0.5377132620007301, new node val ap: 0.6554033501512493
INFO:root:start 1 epoch
INFO:root:epoch: 1 took 0.27s
INFO:root:Epoch mean loss: 1.3933969140052795
INFO:root:val auc: 0.5192422138464591, new node val auc: 0.6734693877551021
INFO:root:val ap: 0.5320020171312927, new node val ap: 0.6618507498693256
INFO:root:start 2 epoch
INFO:root:epoch: 2 took 0.23s
INFO:root:Epoch mean loss: 1.3857935667037964
INFO:root:val auc: 0.5146796270581233, new node val auc: 0.6683673469387755
INFO:root:val ap: 0.5201622800065608, new node val ap: 0.6331872075293129
INFO:root:start 3 epoch
INFO:root:epoch: 3 took 0.27s
INFO:root:Epoch mean loss: 1.3880739212036133
INFO:root:val auc: 0.5160682404284865, new node val auc: 0.7091836734693

In [40]:
import torch.nn.functional as F
import math
import logging
import time
import sys
import argparse
import torch
import numpy as np
import pickle
from pathlib import Path

from tgn.evaluation.evaluation import eval_edge_prediction
from tgn.model.tgn import TGN
from tgn.utils.utils import EarlyStopMonitor, RandEdgeSampler, get_neighbor_finder
from tgn.utils.data_processing import get_data, compute_time_statistics

import importlib
import sys
importlib.reload(sys.modules['tgn.utils.data_processing'])

import tgn.model.tgn as tgn
importlib.reload(tgn)
TGN = tgn.TGN

import tgn.utils.utils as utils

importlib.reload(utils)
get_neighbor_finder = utils.get_neighbor_finder

torch.manual_seed(0)
np.random.seed(0)

BATCH_SIZE = 256
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_EPOCH = 100
NUM_HEADS = 4
DROP_OUT = 0.1
data = "p0.8_mu0.1_1"
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 128
TIME_DIM = 1
USE_MEMORY = False
MESSAGE_DIM = 128
MEMORY_DIM = 128
uniform = False
prefix = ""
backprop_every = 1
message_function = 'mlp'
use_source_embedding_in_message= True
use_destination_embedding_in_message = True 
aggregator = 'last'
different_new_nodes = False
randomize_features = True
n_runs = 1 
memory_update_at_end = False 
embedding_module = 'graph_attention' # graph_attention, graph_sum, time, identity
dyrep = True 
memory_updater = "gru"
patience = 3
bidirection = True

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{data}-{epoch}.pth'

### set up logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)

### Extract data for training, validation and testing
node_features, edge_features, full_data, train_data, val_data, test_data, new_node_val_data, \
new_node_test_data = get_data(data,
                              different_new_nodes_between_val_and_test=different_new_nodes, randomize_features=randomize_features)

# Initialize training neighbor finder to retrieve temporal graph
train_ngh_finder = get_neighbor_finder(train_data, uniform)

# Initialize validation and test neighbor finder to retrieve temporal graph
full_ngh_finder = get_neighbor_finder(full_data, uniform)

# Initialize negative samplers. Set seeds for validation and testing so negatives are the same
# across different runs
# NB: in the inductive setting, negatives are sampled only amongst other new nodes
train_rand_sampler = RandEdgeSampler(train_data.sources, train_data.destinations)
val_rand_sampler = RandEdgeSampler(full_data.sources, full_data.destinations, seed=0)
nn_val_rand_sampler = RandEdgeSampler(new_node_val_data.sources, new_node_val_data.destinations, seed=1)
test_rand_sampler = RandEdgeSampler(full_data.sources, full_data.destinations, seed=2)
nn_test_rand_sampler = RandEdgeSampler(new_node_test_data.sources,
                                       new_node_test_data.destinations,
                                       seed=3)

device = 'cuda' if torch.cuda.is_available() else 'mps'


def build_pooled_neighbor_summary(
    ngh_finder,
    node_features_np,
    source_nodes,
    timestamps,
    n_neighbors,
    device,
):
    """
    对每个 source u 在时间 t，取 t 之前的 temporal neighbors，
    用这些邻居的 raw node features 做 mean pooling，得到 summary。

    返回:
      summary_tensor: [B, D]
      valid_mask: [B]，表示该样本是否至少有一个历史邻居
    """
    source_nodes = np.asarray(source_nodes, dtype=np.int64)
    timestamps = np.asarray(timestamps, dtype=np.float64)

    neighbors, edge_idxs, edge_times = ngh_finder.get_temporal_neighbor_bidirection(
        source_nodes,
        timestamps,
        n_neighbors=n_neighbors
    )
    # neighbors: [B, n_neighbors], padding 一般是 0
    # edge_times: [B, n_neighbors]

    B = neighbors.shape[0]
    D = node_features_np.shape[1]
    summary = np.zeros((B, D), dtype=np.float32)
    valid_mask = np.zeros(B, dtype=bool)

    for i in range(B):
        nbrs = neighbors[i]
        ts = edge_times[i]

        # 只保留真实历史邻居
        valid = ts > 0
        nbrs = nbrs[valid]

        # 去掉 padding（有些实现 padding 节点 id 也是 0）
        nbrs = nbrs[nbrs >= 0]

        if len(nbrs) == 0:
            continue

        valid_mask[i] = True
        summary[i] = node_features_np[nbrs].mean(axis=0)

    summary_tensor = torch.from_numpy(summary).to(device)
    valid_mask = torch.from_numpy(valid_mask).to(device)

    return summary_tensor, valid_mask


def neighborhood_summary_infonce_loss(
    z_u,
    pooled_summary,
    valid_mask,
    query_proj,
    summary_proj,
    temperature=0.2,
):
    """
    z_u: [B, D]，TGN 产生的 source 时序表示
    pooled_summary: [B, D]，时间邻域的 mean-pooled summary
    valid_mask: [B]
    """
    if valid_mask.sum() == 0:
        return torch.tensor(0.0, device=z_u.device, requires_grad=True)

    z_u = z_u[valid_mask]
    pooled_summary = pooled_summary[valid_mask]

    q = query_proj(z_u)
    k = summary_proj(pooled_summary)

    q = torch.nn.functional.normalize(q, dim=1)
    k = torch.nn.functional.normalize(k, dim=1)

    logits = q @ k.t() / temperature   # [M, M]
    labels = torch.arange(logits.size(0), device=logits.device)

    loss = torch.nn.functional.cross_entropy(logits, labels)
    return loss


def compute_src_dst_temporal_embeddings(
    tgn,
    sources_batch,
    destinations_batch,
    negatives_batch,
    timestamps_batch,
    edge_idxs_batch,
    n_neighbors,
):
    """
    复用标准 TGN 的 compute_temporal_embeddings，
    同时返回 source / destination 的时序表示
    """
    source_embedding, destination_embedding, negative_embedding = tgn.compute_temporal_embeddings(
        source_nodes=sources_batch,
        destination_nodes=destinations_batch,
        negative_nodes=negatives_batch,
        edge_times=timestamps_batch,
        edge_idxs=edge_idxs_batch,
        n_neighbors=n_neighbors
    )
    return source_embedding, destination_embedding


def eval_summary_retrieval(
    model,
    query_proj,
    summary_proj,
    ngh_finder,
    data,
    rand_sampler,
    node_features_np,
    batch_size,
    n_neighbors,
    device,
    temperature=0.2,
    topk=5,
):
    """
    评估：
      - Top-1 summary retrieval accuracy
      - Top-k accuracy
      - MRR

    query: z_u(t)
    positive key: pooled neighborhood summary of the same sample
    negatives: other summaries in the same batch
    """
    model.eval()
    query_proj.eval()
    summary_proj.eval()

    num_instance = len(data.sources)
    num_batch = math.ceil(num_instance / batch_size)

    total_valid = 0
    total_top1 = 0.0
    total_topk = 0.0
    total_mrr = 0.0

    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]

            size = len(sources_batch)
            if size == 0:
                continue

            _, negatives_batch = rand_sampler.sample(size)
            # src / dst embeddings
            src_emb, dst_emb = compute_src_dst_temporal_embeddings(
                tgn=model,
                sources_batch=sources_batch,
                destinations_batch=destinations_batch,
                negatives_batch=negatives_batch,
                timestamps_batch=timestamps_batch,
                edge_idxs_batch=edge_idxs_batch,
                n_neighbors=n_neighbors
            )

            # source summaries
            src_summary, src_valid_mask = build_pooled_neighbor_summary(
                ngh_finder=ngh_finder,
                node_features_np=node_features_np,
                source_nodes=sources_batch,
                timestamps=timestamps_batch,
                n_neighbors=n_neighbors,
                device=device,
            )

            # destination summaries
            dst_summary, dst_valid_mask = build_pooled_neighbor_summary(
                ngh_finder=ngh_finder,
                node_features_np=node_features_np,
                source_nodes=destinations_batch,
                timestamps=timestamps_batch,
                n_neighbors=n_neighbors,
                device=device,
            )

            # 拼起来一起评估
            emb_list = []
            sum_list = []

            if src_valid_mask.sum().item() > 0:
                emb_list.append(src_emb[src_valid_mask])
                sum_list.append(src_summary[src_valid_mask])

            if dst_valid_mask.sum().item() > 0:
                emb_list.append(dst_emb[dst_valid_mask])
                sum_list.append(dst_summary[dst_valid_mask])

            if len(emb_list) == 0:
                continue

            z_u = torch.cat(emb_list, dim=0)
            pooled_summary = torch.cat(sum_list, dim=0)

            q = query_proj(z_u)
            s = summary_proj(pooled_summary)

            q = F.normalize(q, dim=1)
            s = F.normalize(s, dim=1)

            logits = q @ s.t() / temperature   # [M, M]
            M = logits.size(0)
            labels = torch.arange(M, device=logits.device)

            # Top-1
            pred_top1 = torch.argmax(logits, dim=1)
            top1_correct = (pred_top1 == labels).float().sum().item()

            # Top-k
            k_eff = min(topk, M)
            topk_idx = torch.topk(logits, k=k_eff, dim=1).indices   # [M, k_eff]
            topk_correct = (topk_idx == labels.unsqueeze(1)).any(dim=1).float().sum().item()

            # MRR
            sorted_idx = torch.argsort(logits, dim=1, descending=True)  # [M, M]
            match_pos = (sorted_idx == labels.unsqueeze(1)).nonzero(as_tuple=False)
            # match_pos[:, 1] 就是每个样本真实 summary 的 rank-1
            ranks = match_pos[:, 1].float() + 1.0
            mrr = (1.0 / ranks).sum().item()

            total_valid += M
            total_top1 += top1_correct
            total_topk += topk_correct
            total_mrr += mrr

    if total_valid == 0:
        return {
            "top1": 0.0,
            "topk": 0.0,
            "mrr": 0.0,
            "num_valid": 0,
        }

    return {
        "top1": total_top1 / total_valid,
        "topk": total_topk / total_valid,
        "mrr": total_mrr / total_valid,
        "num_valid": total_valid,
    }


# Compute time statistics
mean_time_shift_src, std_time_shift_src, mean_time_shift_dst, std_time_shift_dst = \
  compute_time_statistics(full_data.sources, full_data.destinations, full_data.timestamps)


results_path = "results/pretrain_summary_retrieval.pkl" if prefix == "" else f"results/{prefix}_pretrain_summary_retrieval.pkl"
Path("results/").mkdir(parents=True, exist_ok=True)

# Initialize Model
tgn = TGN(neighbor_finder=train_ngh_finder, node_features=node_features,
          edge_features=edge_features, device=device,
          n_layers=NUM_LAYER,
          n_heads=NUM_HEADS, dropout=DROP_OUT, use_memory=USE_MEMORY,
          message_dimension=MESSAGE_DIM, memory_dimension=MEMORY_DIM,
          memory_update_at_start=not memory_update_at_end,
          embedding_module_type=embedding_module,
          message_function=message_function,
          aggregator_type=aggregator,
          memory_updater_type=memory_updater,
          n_neighbors=NUM_NEIGHBORS,
          mean_time_shift_src=mean_time_shift_src, std_time_shift_src=std_time_shift_src,
          mean_time_shift_dst=mean_time_shift_dst, std_time_shift_dst=std_time_shift_dst,
          use_destination_embedding_in_message=use_destination_embedding_in_message,
          use_source_embedding_in_message=use_source_embedding_in_message,
          dyrep=dyrep, bidirection=bidirection )
tgn = tgn.to(device)

proj_dim = node_features.shape[1]   # 和 TGN 输出维度先保持一致
query_proj = torch.nn.Sequential(
    torch.nn.Linear(node_features.shape[1], proj_dim),
    torch.nn.ReLU(),
    torch.nn.Linear(proj_dim, proj_dim),
).to(device)

summary_proj = torch.nn.Sequential(
    torch.nn.Linear(node_features.shape[1], proj_dim),
    torch.nn.ReLU(),
    torch.nn.Linear(proj_dim, proj_dim),
).to(device)

temperature = 0.2

optimizer = torch.optim.AdamW(
    list(tgn.parameters()) + list(query_proj.parameters()) + list(summary_proj.parameters()),
    lr=LEARNING_RATE
)

num_instance = len(train_data.sources)
num_batch = math.ceil(num_instance / BATCH_SIZE)

logger.info('num of training instances: {}'.format(num_instance))
logger.info('num of batches per epoch: {}'.format(num_batch))
idx_list = np.arange(num_instance)

val_top1s = []
val_topks = []
val_mrrs = []

new_nodes_val_top1s = []
new_nodes_val_topks = []
new_nodes_val_mrrs = []

epoch_times = []
total_epoch_times = []
train_losses = []

early_stopper = EarlyStopMonitor(max_round=patience)
for epoch in range(NUM_EPOCH):
  start_epoch = time.time()
  ### Training

  # Reinitialize memory of the model at the start of each epoch
  if USE_MEMORY:
    tgn.memory.__init_memory__()

  # Train using only training graph
  tgn.set_neighbor_finder(train_ngh_finder)
  m_loss = []

  logger.info('start {} epoch'.format(epoch))
  for k in range(0, num_batch, backprop_every):
    loss = 0
    optimizer.zero_grad()

    # Custom loop to allow to perform backpropagation only every a certain number of batches
    for j in range(backprop_every):
      batch_idx = k + j

      if batch_idx >= num_batch:
        continue

      start_idx = batch_idx * BATCH_SIZE
      end_idx = min(num_instance, start_idx + BATCH_SIZE)
      sources_batch, destinations_batch = train_data.sources[start_idx:end_idx], \
                                          train_data.destinations[start_idx:end_idx]
      edge_idxs_batch = train_data.edge_idxs[start_idx: end_idx]
      timestamps_batch = train_data.timestamps[start_idx:end_idx]

      size = len(sources_batch)
      _, negatives_batch = train_rand_sampler.sample(size)

      tgn = tgn.train()
      # 1) 同时得到 source / destination 的时序表示
      src_emb, dst_emb = compute_src_dst_temporal_embeddings(
          tgn=tgn,
          sources_batch=sources_batch,
          destinations_batch=destinations_batch,
          negatives_batch=negatives_batch,
          timestamps_batch=timestamps_batch,
          edge_idxs_batch=edge_idxs_batch,
          n_neighbors=NUM_NEIGHBORS
      )   # [B, D], [B, D]

      # 2) source 的时间邻域 summary
      src_summary, src_valid_mask = build_pooled_neighbor_summary(
          ngh_finder=train_ngh_finder,
          node_features_np=node_features,
          source_nodes=sources_batch,
          timestamps=timestamps_batch,
          n_neighbors=NUM_NEIGHBORS,
          device=device,
      )

      # 3) destination 的时间邻域 summary
      dst_summary, dst_valid_mask = build_pooled_neighbor_summary(
          ngh_finder=train_ngh_finder,
          node_features_np=node_features,
          source_nodes=destinations_batch,
          timestamps=timestamps_batch,
          n_neighbors=NUM_NEIGHBORS,
          device=device,
      )

      # 4) source 侧 InfoNCE
      src_loss = neighborhood_summary_infonce_loss(
          z_u=src_emb,
          pooled_summary=src_summary,
          valid_mask=src_valid_mask,
          query_proj=query_proj,
          summary_proj=summary_proj,
          temperature=temperature,
      )

      # 5) destination 侧 InfoNCE
      dst_loss = neighborhood_summary_infonce_loss(
          z_u=dst_emb,
          pooled_summary=dst_summary,
          valid_mask=dst_valid_mask,
          query_proj=query_proj,
          summary_proj=summary_proj,
          temperature=temperature,
      )

      # 6) 总 loss
      loss += 0.5 * (src_loss + dst_loss)

    loss /= backprop_every

    loss.backward()
    optimizer.step()
    m_loss.append(loss.item())

    # Detach memory after 'backprop_every' number of batches so we don't backpropagate to
    # the start of time
    if USE_MEMORY:
      tgn.memory.detach_memory()

  epoch_time = time.time() - start_epoch
  epoch_times.append(epoch_time)

  tgn.set_neighbor_finder(full_ngh_finder)

  if USE_MEMORY:
    train_memory_backup = tgn.memory.backup_memory()

  val_metrics = eval_summary_retrieval(
      model=tgn,
      query_proj=query_proj,
      summary_proj=summary_proj,
      ngh_finder=full_ngh_finder,
      data=val_data,
      rand_sampler=val_rand_sampler,
      node_features_np=node_features,
      batch_size=BATCH_SIZE,
      n_neighbors=NUM_NEIGHBORS,
      device=device,
      temperature=temperature,
      topk=5,
  )

  if USE_MEMORY:
    val_memory_backup = tgn.memory.backup_memory()
    tgn.memory.restore_memory(train_memory_backup)

  nn_val_metrics = eval_summary_retrieval(
      model=tgn,
      query_proj=query_proj,
      summary_proj=summary_proj,
      ngh_finder=full_ngh_finder,
      data=new_node_val_data,
      rand_sampler=nn_val_rand_sampler,
      node_features_np=node_features,
      batch_size=BATCH_SIZE,
      n_neighbors=NUM_NEIGHBORS,
      device=device,
      temperature=temperature,
      topk=5,
  )

  if USE_MEMORY:
    tgn.memory.restore_memory(val_memory_backup)

  val_top1s.append(val_metrics["top1"])
  val_topks.append(val_metrics["topk"])
  val_mrrs.append(val_metrics["mrr"])

  new_nodes_val_top1s.append(nn_val_metrics["top1"])
  new_nodes_val_topks.append(nn_val_metrics["topk"])
  new_nodes_val_mrrs.append(nn_val_metrics["mrr"])

  train_losses.append(np.mean(m_loss))
  # Save temporary results to disk
  pickle.dump({
    "val_top1s": val_top1s,
    "val_topks": val_topks,
    "val_mrrs": val_mrrs,
    "new_nodes_val_top1s": new_nodes_val_top1s,
    "new_nodes_val_topks": new_nodes_val_topks,
    "new_nodes_val_mrrs": new_nodes_val_mrrs,
    "train_losses": train_losses,
    "epoch_times": epoch_times,
    "total_epoch_times": total_epoch_times
  }, open(results_path, "wb"))

  total_epoch_time = time.time() - start_epoch
  total_epoch_times.append(total_epoch_time)

  logger.info('epoch: {} took {:.2f}s'.format(epoch, total_epoch_time))
  logger.info('Epoch mean loss: {}'.format(np.mean(m_loss)))
  logger.info(
    'val top1: {:.4f}, val top5: {:.4f}, val mrr: {:.4f}'.format(
        val_metrics["top1"], val_metrics["topk"], val_metrics["mrr"])
  )
  logger.info(
    'new node val top1: {:.4f}, new node val top5: {:.4f}, new node val mrr: {:.4f}'.format(
        nn_val_metrics["top1"], nn_val_metrics["topk"], nn_val_metrics["mrr"])
  )

  # Early stopping
  if early_stopper.early_stop_check(val_metrics["mrr"]):
    logger.info('No improvement over {} epochs, stop training'.format(early_stopper.max_round))
    logger.info(f'Loading the best model at epoch {early_stopper.best_epoch}')
    best_model_path = get_checkpoint_path(early_stopper.best_epoch)
    tgn.load_state_dict(torch.load(best_model_path))
    logger.info(f'Loaded the best model at epoch {early_stopper.best_epoch} for inference')
    tgn.eval()
    break
  else:
    torch.save(tgn.state_dict(), get_checkpoint_path(epoch))

# Training has finished, we have loaded the best model, and we want to backup its current
# memory (which has seen validation edges) so that it can also be used when testing on unseen
# nodes
if USE_MEMORY:
  val_memory_backup = tgn.memory.backup_memory()

### Test
tgn.embedding_module.neighbor_finder = full_ngh_finder

test_metrics = eval_summary_retrieval(
    model=tgn,
    query_proj=query_proj,
    summary_proj=summary_proj,
    ngh_finder=full_ngh_finder,
    data=test_data,
    rand_sampler=test_rand_sampler,
    node_features_np=node_features,
    batch_size=BATCH_SIZE,
    n_neighbors=NUM_NEIGHBORS,
    device=device,
    temperature=temperature,
    topk=5,
)

if USE_MEMORY:
  tgn.memory.restore_memory(val_memory_backup)

nn_test_metrics = eval_summary_retrieval(
    model=tgn,
    query_proj=query_proj,
    summary_proj=summary_proj,
    ngh_finder=full_ngh_finder,
    data=new_node_test_data,
    rand_sampler=nn_test_rand_sampler,
    node_features_np=node_features,
    batch_size=BATCH_SIZE,
    n_neighbors=NUM_NEIGHBORS,
    device=device,
    temperature=temperature,
    topk=5,
)

logger.info(
  'Test statistics: Old nodes -- top1: {:.4f}, top5: {:.4f}, mrr: {:.4f}'.format(
      test_metrics["top1"], test_metrics["topk"], test_metrics["mrr"])
)
logger.info(
  'Test statistics: New nodes -- top1: {:.4f}, top5: {:.4f}, mrr: {:.4f}'.format(
      nn_test_metrics["top1"], nn_test_metrics["topk"], nn_test_metrics["mrr"])
)

pickle.dump({
  "val_top1s": val_top1s,
  "val_topks": val_topks,
  "val_mrrs": val_mrrs,
  "new_nodes_val_top1s": new_nodes_val_top1s,
  "new_nodes_val_topks": new_nodes_val_topks,
  "new_nodes_val_mrrs": new_nodes_val_mrrs,
  "test_top1": test_metrics["top1"],
  "test_top5": test_metrics["topk"],
  "test_mrr": test_metrics["mrr"],
  "new_node_test_top1": nn_test_metrics["top1"],
  "new_node_test_top5": nn_test_metrics["topk"],
  "new_node_test_mrr": nn_test_metrics["mrr"],
  "epoch_times": epoch_times,
  "train_losses": train_losses,
  "total_epoch_times": total_epoch_times
}, open(results_path, "wb"))

logger.info('Saving TGN model')
if USE_MEMORY:
  # Restore memory at the end of validation (save a model which is ready for testing)
  tgn.memory.restore_memory(val_memory_backup)
torch.save(tgn.state_dict(), MODEL_SAVE_PATH)
logger.info('TGN model saved')


INFO:root:num of training instances: 274
INFO:root:num of batches per epoch: 2
INFO:root:start 0 epoch


The dataset has 472 interactions, involving 50 different nodes
The training dataset has 274 interactions, involving 45 different nodes
The validation dataset has 71 interactions, involving 48 different nodes
The test dataset has 71 interactions, involving 48 different nodes
The new node validation dataset has 14 interactions, involving 14 different nodes
The new node test dataset has 7 interactions, involving 11 different nodes
5 nodes were used for the inductive testing, i.e. are never seen during training


INFO:root:epoch: 0 took 1.16s
INFO:root:Epoch mean loss: 4.214629411697388
INFO:root:val top1: 0.0070, val top5: 0.0493, val mrr: 0.0458
INFO:root:new node val top1: 0.0714, new node val top5: 0.2143, new node val mrr: 0.1782
INFO:root:start 1 epoch
INFO:root:epoch: 1 took 0.30s
INFO:root:Epoch mean loss: 4.204206943511963
INFO:root:val top1: 0.0070, val top5: 0.0563, val mrr: 0.0475
INFO:root:new node val top1: 0.0714, new node val top5: 0.2500, new node val mrr: 0.1826
INFO:root:start 2 epoch
INFO:root:epoch: 2 took 0.30s
INFO:root:Epoch mean loss: 4.195883274078369
INFO:root:val top1: 0.0141, val top5: 0.0775, val mrr: 0.0577
INFO:root:new node val top1: 0.0357, new node val top5: 0.2500, new node val mrr: 0.1520
INFO:root:start 3 epoch
INFO:root:epoch: 3 took 0.30s
INFO:root:Epoch mean loss: 4.185172080993652
INFO:root:val top1: 0.0211, val top5: 0.0986, val mrr: 0.0695
INFO:root:new node val top1: 0.0714, new node val top5: 0.2500, new node val mrr: 0.1833
INFO:root:start 4 epoch


In [42]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans

from tgn.utils.data_processing import get_data
from tgn.utils.utils import get_neighbor_finder
from tgn.model.tgn import TGN


# =========================================================
# 1. 基本配置
# =========================================================
FILE_NAME = "p0.8_mu0.1_1"
MODEL_PATH = f"saved_models/{FILE_NAME}.pth"
NODE2ID_PATH = f"preprocess/{FILE_NAME}_node2id.csv"
OUT_DIR = Path(f"results/{FILE_NAME}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "mps")

# ===== 这些超参数必须和训练时一致 =====
BATCH_SIZE = 256
NUM_NEIGHBORS = 20
NUM_LAYER = 2
NUM_HEADS = 4
DROP_OUT = 0.1
USE_MEMORY = True
MESSAGE_DIM = 128
MEMORY_DIM = 128
embedding_module = "graph_attention"
message_function = "mlp"
aggregator = "last"
memory_updater = "gru"
use_destination_embedding_in_message = False
use_source_embedding_in_message = False
dyrep = True
bidirection = True

K = 5


# =========================================================
# 1.5 读取 node2id，并构造 id -> node 映射
# =========================================================
node2id_df = pd.read_csv(NODE2ID_PATH)

# 兼容两种列顺序：["node", "id"] 或 ["id", "node"]
if {"node", "id"}.issubset(node2id_df.columns):
    id_to_node = dict(zip(node2id_df["id"], node2id_df["node"]))
elif node2id_df.shape[1] >= 2:
    col0, col1 = node2id_df.columns[:2]

    # 尝试自动判断哪列是 id
    if "id" in col0.lower():
        id_col, node_col = col0, col1
    elif "id" in col1.lower():
        id_col, node_col = col1, col0
    else:
        # 默认第二列是 id，第一列是 node
        node_col, id_col = col0, col1

    id_to_node = dict(zip(node2id_df[id_col], node2id_df[node_col]))
else:
    raise ValueError(f"Unexpected node2id file format: {NODE2ID_PATH}")

print(f"Loaded node2id from {NODE2ID_PATH}, total {len(id_to_node)} nodes")


# =========================================================
# 2. 读取数据
# =========================================================
node_features, edge_features, full_data, train_data, val_data, test_data, \
new_node_val_data, new_node_test_data = get_data(FILE_NAME)

full_ngh_finder = get_neighbor_finder(full_data, uniform=False)

sources = full_data.sources
destinations = full_data.destinations
timestamps = full_data.timestamps

last_timestamp_sources = dict()
last_timestamp_destinations = dict()

all_timediffs_src = []
all_timediffs_dst = []

for k in range(len(sources)):
    s = sources[k]
    d = destinations[k]
    t = timestamps[k]

    all_timediffs_src.append(t - last_timestamp_sources.get(s, 0))
    all_timediffs_dst.append(t - last_timestamp_destinations.get(d, 0))

    last_timestamp_sources[s] = t
    last_timestamp_destinations[d] = t

mean_time_shift_src = np.mean(all_timediffs_src)
std_time_shift_src = np.std(all_timediffs_src)
mean_time_shift_dst = np.mean(all_timediffs_dst)
std_time_shift_dst = np.std(all_timediffs_dst)

print("mean_time_shift_src =", mean_time_shift_src)
print("std_time_shift_src  =", std_time_shift_src)
print("mean_time_shift_dst =", mean_time_shift_dst)
print("std_time_shift_dst  =", std_time_shift_dst)


# =========================================================
# 3. 重建模型
# =========================================================
tgn = TGN(
    neighbor_finder=full_ngh_finder,
    node_features=node_features,
    edge_features=edge_features,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    memory_update_at_start=True,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator,
    memory_updater_type=memory_updater,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift_src=mean_time_shift_src,
    std_time_shift_src=std_time_shift_src,
    mean_time_shift_dst=mean_time_shift_dst,
    std_time_shift_dst=std_time_shift_dst,
    use_destination_embedding_in_message=use_destination_embedding_in_message,
    use_source_embedding_in_message=use_source_embedding_in_message,
    dyrep=dyrep,
    bidirection=bidirection,
).to(device)

ckpt = torch.load(MODEL_PATH, map_location=device)

if isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

tgn.load_state_dict(state_dict, strict=False)
tgn.eval()

if USE_MEMORY and hasattr(tgn, "memory") and tgn.memory is not None:
    if hasattr(tgn.memory, "__init_memory__"):
        tgn.memory.__init_memory__()
    elif hasattr(tgn.memory, "reset_memory"):
        tgn.memory.reset_memory()


# =========================================================
# 4. 导出 node-event embedding
# =========================================================
all_src_emb = []
all_dst_emb = []
all_meta = []

num_instance = len(full_data.sources)
num_batch = int(np.ceil(num_instance / BATCH_SIZE))

with torch.no_grad():
    for batch_idx in range(num_batch):
        start = batch_idx * BATCH_SIZE
        end = min(num_instance, start + BATCH_SIZE)

        sources_batch = full_data.sources[start:end]
        destinations_batch = full_data.destinations[start:end]
        timestamps_batch = full_data.timestamps[start:end]
        edge_idxs_batch = full_data.edge_idxs[start:end]

        negative_nodes_batch = destinations_batch

        source_node_embedding, destination_node_embedding, _ = tgn.compute_temporal_embeddings(
            source_nodes=sources_batch,
            destination_nodes=destinations_batch,
            negative_nodes=negative_nodes_batch,
            edge_times=timestamps_batch,
            edge_idxs=edge_idxs_batch,
            n_neighbors=NUM_NEIGHBORS
        )

        src_emb_np = source_node_embedding.detach().cpu().numpy().astype(np.float32)
        dst_emb_np = destination_node_embedding.detach().cpu().numpy().astype(np.float32)

        all_src_emb.append(src_emb_np)
        all_dst_emb.append(dst_emb_np)

        batch_meta = pd.DataFrame({
            "source_id": sources_batch,
            "destination_id": destinations_batch,
            "timestamp": timestamps_batch,
            "edge_idx": edge_idxs_batch
        })
        all_meta.append(batch_meta)

        if batch_idx % 20 == 0 or batch_idx == num_batch - 1:
            print(f"[{batch_idx + 1}/{num_batch}] done")

src_emb = np.concatenate(all_src_emb, axis=0)
dst_emb = np.concatenate(all_dst_emb, axis=0)
meta = pd.concat(all_meta, axis=0, ignore_index=True)

print("src_emb shape:", src_emb.shape)
print("dst_emb shape:", dst_emb.shape)
print("meta shape   :", meta.shape)

# 把内部 id 映射回原始 node
meta["source"] = meta["source_id"].map(id_to_node)
meta["destination"] = meta["destination_id"].map(id_to_node)

# 检查是否有没映射上的
if meta["source"].isna().any() or meta["destination"].isna().any():
    bad_src = meta.loc[meta["source"].isna(), "source_id"].unique().tolist()
    bad_dst = meta.loc[meta["destination"].isna(), "destination_id"].unique().tolist()
    raise ValueError(
        f"Some node ids cannot be mapped back.\n"
        f"Bad source ids: {bad_src[:10]}\n"
        f"Bad destination ids: {bad_dst[:10]}"
    )

np.save(OUT_DIR / "full_src.npy", src_emb)
np.save(OUT_DIR / "full_dst.npy", dst_emb)

# 保存一个带原始 node 的 meta
meta[["source", "destination", "timestamp", "edge_idx"]].to_csv(
    OUT_DIR / "node_event_meta.csv", index=False
)

print("Saved:")
print(OUT_DIR / "full_src.npy")
print(OUT_DIR / "full_dst.npy")
print(OUT_DIR / "node_event_meta.csv")


# =========================================================
# 5. KMeans 聚类
# =========================================================
X = np.concatenate([src_emb, dst_emb], axis=0)
print("KMeans input shape:", X.shape)

kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
labels = kmeans.fit_predict(X)

N = len(meta)
src_labels = labels[:N]
dst_labels = labels[N:]

result_df = meta[["source", "destination", "timestamp"]].copy()
result_df["source_commu"] = src_labels
result_df["destination_commu"] = dst_labels

result_df.to_csv(OUT_DIR / "tgn_kmeans.csv", index=False)

print("Saved clustering result to:")
print(OUT_DIR / "tgn_kmeans.csv")
print(result_df.head())

Loaded node2id from preprocess/p0.8_mu0.1_1_node2id.csv, total 50 nodes
The dataset has 472 interactions, involving 50 different nodes
The training dataset has 274 interactions, involving 45 different nodes
The validation dataset has 71 interactions, involving 48 different nodes
The test dataset has 71 interactions, involving 48 different nodes
The new node validation dataset has 14 interactions, involving 14 different nodes
The new node test dataset has 7 interactions, involving 11 different nodes
5 nodes were used for the inductive testing, i.e. are never seen during training
mean_time_shift_src = 226.8072033898305
std_time_shift_src  = 276.62642869004054
mean_time_shift_dst = 250.01271186440678
std_time_shift_dst  = 266.5697825916337
[1/2] done
[2/2] done
src_emb shape: (472, 128)
dst_emb shape: (472, 128)
meta shape   : (472, 4)
Saved:
results/p0.8_mu0.1_1/full_src.npy
results/p0.8_mu0.1_1/full_dst.npy
results/p0.8_mu0.1_1/node_event_meta.csv
KMeans input shape: (944, 128)
Saved cl